# M2: Cosmos Reason Captioning — AV Video Captioning

**Pipeline Position:** Stage 3: Data Curation — AV Video Captioning (Cosmos Reason)  
**Input S3 Path:** `s3://av30lab-user-workspace-{account_id}/users/{profile}/m1/`  
**Output S3 Path:** `s3://av30lab-user-workspace-{account_id}/users/{profile}/m2/captions.json`  
**Source:** [cosmos-cookbook/post_training/reason1/av_video_caption_vqa](https://github.com/NVIDIA/Cosmos/tree/main/cosmos-cookbook/post_training/reason1/av_video_caption_vqa)  
**Instance:** ml.g5.12xlarge (4x NVIDIA A10G, 96GB total VRAM)

In [ ]:
"""Environment Setup"""
import os
import json
import time
import glob
from pathlib import Path
from datetime import datetime, timezone

import boto3
import torch
from PIL import Image
from IPython.display import display, Image as IPImage

# --- S3 Path Configuration ---
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
PROFILE = os.environ.get("USER_PROFILE", "default")

USER_BUCKET = os.environ.get("USER_BUCKET", f"av30lab-user-workspace-{ACCOUNT_ID}")
SHARED_BUCKET = os.environ.get("SHARED_BUCKET", f"av30lab-shared-data-{ACCOUNT_ID}")
INPUT_PREFIX = f"users/{PROFILE}/m1/"
OUTPUT_PREFIX = f"users/{PROFILE}/m2/"
MODEL_CACHE_PREFIX = "model-cache/cosmos-reason1/"

# Local paths
LOCAL_MODEL_DIR = Path("/tmp/cosmos-reason1-7b")
LOCAL_DATA_DIR = Path("/tmp/nuscenes-mini")
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# --- Clients ---
s3 = boto3.client("s3")

print(f"Account ID: {ACCOUNT_ID}")
print(f"Profile: {PROFILE}")
print(f"Input: s3://{USER_BUCKET}/{INPUT_PREFIX}")
print(f"Output: s3://{USER_BUCKET}/{OUTPUT_PREFIX}")
print(f"Model cache: s3://{SHARED_BUCKET}/{MODEL_CACHE_PREFIX}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
"""Pre-flight GPU Check — Require 96GB VRAM (g5.12xlarge: 4x A10G)"""
import subprocess

REQUIRED_VRAM_GB = 96
RECOMMENDED_INSTANCE = "ml.g5.12xlarge"

def check_gpu():
    """Validate GPU availability and VRAM."""
    if not torch.cuda.is_available():
        print("ERROR: No GPU detected!")
        print(f"This notebook requires {REQUIRED_VRAM_GB}GB VRAM.")
        print(f"Recommended instance: {RECOMMENDED_INSTANCE}")
        return False

    # Run nvidia-smi
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True
    )
    print("GPU Status:")
    print("=" * 60)

    total_vram_mb = 0
    for i, line in enumerate(result.stdout.strip().split("\n")):
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 3:
            name, total_mb, free_mb = parts[0], int(parts[1]), int(parts[2])
            total_vram_mb += int(total_mb)
            print(f"  GPU {i}: {name} | Total: {int(total_mb)/1024:.1f}GB | Free: {int(free_mb)/1024:.1f}GB")

    total_vram_gb = total_vram_mb / 1024
    print(f"\nTotal VRAM: {total_vram_gb:.1f}GB")
    print(f"Required:   {REQUIRED_VRAM_GB}GB")

    if total_vram_gb < REQUIRED_VRAM_GB * 0.9:  # 10% tolerance
        print(f"\nWARNING: Insufficient VRAM ({total_vram_gb:.1f}GB < {REQUIRED_VRAM_GB}GB)")
        print(f"Recommendation: Switch to {RECOMMENDED_INSTANCE} (4x A10G = 96GB)")
        return False

    print("\nGPU check PASSED.")
    return True

gpu_ok = check_gpu()
if not gpu_ok:
    raise RuntimeError(
        f"Insufficient GPU resources. Requires {REQUIRED_VRAM_GB}GB VRAM. "
        f"Use instance: {RECOMMENDED_INSTANCE}"
    )

In [ ]:
"""Load Cosmos Reason 1 model from S3 cache (NOT from HuggingFace directly)"""
from transformers import AutoModelForImageTextToText, AutoProcessor, AutoTokenizer

MODEL_ID = "nvidia/Cosmos-Reason1-7B"

# Step 1: Sync model from S3 cache to local
print(f"Downloading model from S3 cache: s3://{SHARED_BUCKET}/{MODEL_CACHE_PREFIX}")
print("This may take 5-10 minutes on first run...")

start_download = time.time()
subprocess.run(
    ["aws", "s3", "sync",
     f"s3://{SHARED_BUCKET}/{MODEL_CACHE_PREFIX}",
     str(LOCAL_MODEL_DIR),
     "--quiet"],
    check=True
)
download_time = time.time() - start_download
print(f"Model download complete in {download_time:.1f}s")

# Step 2: Load model with multi-GPU support
print("\nLoading model into GPU memory...")
start_load = time.time()

processor = AutoProcessor.from_pretrained(
    str(LOCAL_MODEL_DIR),
    trust_remote_code=True
)

model = AutoModelForImageTextToText.from_pretrained(
    str(LOCAL_MODEL_DIR),
    dtype=torch.bfloat16,
    device_map="auto",  # Spread across all 4 GPUs
    trust_remote_code=True,
)
model.eval()

load_time = time.time() - start_load
print(f"Model loaded in {load_time:.1f}s")
print(f"Model device map: {model.hf_device_map if hasattr(model, 'hf_device_map') else 'single device'}")

# Check memory usage
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    print(f"  GPU {i}: {allocated:.1f}GB allocated")

In [ ]:
"""Process nuScenes driving frames — caption an even sample across scenes"""

# ---------------------------------------------------------------------------
# How many frames to caption. Captioning is the GPU-heavy step (~3-8s/frame on
# Cosmos-Reason1-7B), so this is the main cost/time lever. The default keeps a
# workshop run fast and cheap while covering ALL scenes; raise it (up to the
# full manifest) if you want a richer experiment and are willing to pay the GPU
# time. Set to None to caption every frame in the manifest.
MAX_CAPTION_FRAMES = 24
# ---------------------------------------------------------------------------

# Load M1 manifest to get the frame list (+ per-frame scene tags if present)
manifest_obj = s3.get_object(Bucket=USER_BUCKET, Key=f"{INPUT_PREFIX}manifest.json")
manifest = json.loads(manifest_obj["Body"].read())

cam_files = manifest["cam_front_files"]
cam_scenes = manifest.get("cam_front_scenes")  # M1 (new) tags each frame's scene
print(f"M1 manifest lists {len(cam_files)} CAM_FRONT frames "
      f"across {len(manifest.get('scenes', []))} scenes")

# Fetch the CAM_FRONT images referenced by the manifest into LOCAL_DATA_DIR,
# preserving the "samples/CAM_FRONT/..." structure so `LOCAL_DATA_DIR / cam_file`
# resolves. Required because /tmp is empty on a fresh instance.
source_bucket = manifest.get("source_bucket", SHARED_BUCKET)
source_prefix = manifest.get("source_prefix", "datasets/nuscenes-mini/")
print(f"Downloading images from s3://{source_bucket}/{source_prefix} ...")
_downloaded = 0
for cam_file in cam_files:
    dest = LOCAL_DATA_DIR / cam_file
    if dest.exists():
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(source_bucket, f"{source_prefix}{cam_file}", str(dest))
    _downloaded += 1
print(f"Downloaded {_downloaded} new image(s); {len(cam_files)} total available locally")


def select_frames_even(files, scenes, limit):
    """Pick `limit` frames spread as evenly as possible across scenes.

    Uses per-frame scene tags when available (round-robin one frame per scene
    until the quota is filled). Falls back to an even stride over `files` when
    tags are absent (older M1 manifests)."""
    if limit is None or limit >= len(files):
        return list(range(len(files)))
    if scenes and len(scenes) == len(files):
        # group frame indices by scene, then round-robin across scenes
        from collections import OrderedDict
        by_scene = OrderedDict()
        for i, sc in enumerate(scenes):
            by_scene.setdefault(sc, []).append(i)
        picked, exhausted = [], False
        while len(picked) < limit and not exhausted:
            exhausted = True
            for idxs in by_scene.values():
                if idxs:
                    picked.append(idxs.pop(0))
                    exhausted = False
                    if len(picked) >= limit:
                        break
        return sorted(picked)
    # fallback: even stride over the full list
    stride = max(1, len(files) // limit)
    return list(range(0, len(files), stride))[:limit]


sel_idx = select_frames_even(cam_files, cam_scenes, MAX_CAPTION_FRAMES)
selected = [(i, cam_files[i], (cam_scenes[i] if cam_scenes else None)) for i in sel_idx]
if cam_scenes:
    from collections import Counter
    _dist = Counter(s for _, _, s in selected)
    print(f"Captioning {len(selected)} frames, even across "
          f"{len(_dist)} scenes: {dict(_dist)}")
else:
    print(f"Captioning {len(selected)} frames (stride sample; "
          f"manifest has no scene tags)")

# Caption generation prompt for AV context
AV_CAPTION_PROMPT = (
    "You are an autonomous vehicle perception system. "
    "Describe this driving scene in detail, including: "
    "1) Road layout and lane configuration, "
    "2) All visible vehicles, pedestrians, and cyclists with their positions, "
    "3) Traffic signals and signs, "
    "4) Weather and lighting conditions, "
    "5) Any potential hazards or safety-relevant observations."
)

def generate_caption(image_path: str, prompt: str = AV_CAPTION_PROMPT) -> str:
    """Generate caption for a single driving scene image."""
    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )

    # Decode only new tokens
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    caption = processor.decode(generated_ids, skip_special_tokens=True)
    return caption.strip()

# Process the selected frames
captions = []
start_inference = time.time()

for n, (frame_idx, cam_file, scene) in enumerate(selected):
    img_path = LOCAL_DATA_DIR / cam_file
    if not img_path.exists():
        print(f"  [{n+1}] SKIP - file not found: {cam_file}")
        continue

    t0 = time.time()
    caption = generate_caption(str(img_path))
    elapsed = time.time() - t0

    captions.append({
        "frame_idx": frame_idx,
        "scene": scene,
        "filename": cam_file,
        "caption": caption,
        "inference_time_s": round(elapsed, 2),
        "timestamp": datetime.now(timezone.utc).isoformat()
    })

    _scene_tag = f"{scene} | " if scene else ""
    print(f"  [{n+1}/{len(selected)}] {elapsed:.1f}s | {_scene_tag}{cam_file}")
    print(f"    Caption: {caption[:120]}...\n")

total_inference_time = time.time() - start_inference
print(f"\nTotal inference time: {total_inference_time:.1f}s for {len(captions)} frames")
print(f"Average per frame: {total_inference_time/max(len(captions),1):.1f}s")

In [ ]:
"""Display sample captions alongside video frames"""
import matplotlib.pyplot as plt

# Show up to 4 captioned frames in a grid
display_count = min(4, len(captions))
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i in range(display_count):
    cap = captions[i]
    img_path = LOCAL_DATA_DIR / cap["filename"]
    if img_path.exists():
        img = Image.open(img_path)
        axes[i].imshow(img)
        # Wrap caption text
        wrapped = cap["caption"][:150] + ("..." if len(cap["caption"]) > 150 else "")
        axes[i].set_title(wrapped, fontsize=8, wrap=True)
    axes[i].axis("off")

# Hide unused subplots
for i in range(display_count, 4):
    axes[i].axis("off")

plt.suptitle("Cosmos Reason 1 — AV Scene Captioning Results", fontsize=14)
plt.tight_layout()
plt.savefig("/tmp/m2_caption_samples.png", dpi=100, bbox_inches="tight")
plt.show()

print(f"\nDisplayed {display_count} captioned frames.")
print("Full captions will be saved to S3 in the next cell.")

In [ ]:
"""Write captions to S3 output path for M3"""

# Build output payload
output_data = {
    "module": "M2_Cosmos_Reason_Captioning",
    "model": MODEL_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "num_captions": len(captions),
    "total_inference_time_s": round(total_inference_time, 2),
    "avg_inference_time_s": round(total_inference_time / max(len(captions), 1), 2),
    "prompt_template": AV_CAPTION_PROMPT,
    "captions": captions
}

# Upload to S3
output_key = f"{OUTPUT_PREFIX}captions.json"
s3.put_object(
    Bucket=USER_BUCKET,
    Key=output_key,
    Body=json.dumps(output_data, indent=2, ensure_ascii=False),
    ContentType="application/json"
)

print(f"Captions written to: s3://{USER_BUCKET}/{output_key}")
print(f"Total captions: {len(captions)}")
print(f"\nNext module: M3_Cosmos_Curator.ipynb will read from this path.")

In [ ]:
"""Cost Analysis — g5.12xlarge at $7.09/hr vs human annotation"""

# Instance pricing
INSTANCE_TYPE = "ml.g5.12xlarge"
INSTANCE_COST_PER_HOUR = 7.09  # USD, us-west-2 on-demand
KRW_RATE = 1370

# Actual execution metrics
model_download_min = download_time / 60
model_load_min = load_time / 60
inference_min = total_inference_time / 60
total_min = model_download_min + model_load_min + inference_min + 5  # +5 min overhead
total_hours = total_min / 60

compute_cost_usd = INSTANCE_COST_PER_HOUR * total_hours
compute_cost_krw = compute_cost_usd * KRW_RATE

# Human annotation comparison
HUMAN_COST_PER_CLIP = 2.50  # USD (industry average for detailed AV annotation)
human_total_cost = HUMAN_COST_PER_CLIP * len(captions)

print("=" * 60)
print("M2 Cosmos Reason Captioning — Cost Analysis")
print("=" * 60)
print(f"Instance type:       {INSTANCE_TYPE}")
print(f"Instance cost:       ${INSTANCE_COST_PER_HOUR:.2f}/hr")
print(f"")
print(f"Breakdown:")
print(f"  Model download:    {model_download_min:.1f} min")
print(f"  Model loading:     {model_load_min:.1f} min")
print(f"  Inference:         {inference_min:.1f} min ({len(captions)} frames)")
print(f"  Overhead:          ~5 min")
print(f"  Total time:        {total_min:.1f} min")
print(f"")
print(f"Compute cost:        ${compute_cost_usd:.2f} USD ({compute_cost_krw:.0f} KRW)")
print(f"")
print(f"--- Comparison: Human Annotation ---")
print(f"Human cost/clip:     ${HUMAN_COST_PER_CLIP:.2f} USD")
print(f"Human total ({len(captions)} clips): ${human_total_cost:.2f} USD")
print(f"AI cost ({len(captions)} clips):     ${compute_cost_usd:.2f} USD")
print(f"Savings:             ${human_total_cost - compute_cost_usd:.2f} USD ({((human_total_cost - compute_cost_usd)/human_total_cost)*100:.0f}% cheaper)")
print(f"")
print(f"At scale (10,000 clips):")
print(f"  Human:  ${10000 * HUMAN_COST_PER_CLIP:,.0f} USD")
est_10k_hours = (total_inference_time / max(len(captions), 1)) * 10000 / 3600
print(f"  AI:     ${est_10k_hours * INSTANCE_COST_PER_HOUR:,.0f} USD ({est_10k_hours:.1f} GPU-hours)")
print("=" * 60)

In [ ]:
"""Mark this module complete on the participant dashboard (best-effort, non-fatal)."""
import sys
from pathlib import Path
for _b in (Path.cwd(), Path.cwd().parent, Path.home()):
    _cand = _b / "scripts" / "av30_progress.py"
    if _cand.exists():
        sys.path.insert(0, str(_b / "scripts"))
        break
try:
    from av30_progress import mark_complete
    mark_complete("m02-cosmos-reason")
except Exception as _e:
    print(f"[progress] helper unavailable ({_e}); skipping — module still complete.")